_ExplainDB — Copyright (C) 2026 Prof. Dr. Jens Dittrich, Saarland University._  
_Licensed under the GNU Affero General Public License v3 (AGPL-3.0). See the `LICENSE` file._

# MVCC Playground

Build schedules of transactions on the MVCC store of this repository and run them operation by operation.

- **Scenario:** choose a classic schedule, or **empty** to build your own. Once you change a scenario, the dropdown
  shows **own schedule**; choose the scenario again to start over.
- **Build:** choose a transaction and an operation (and a key and values where they apply), then click **add**.
  Only operations that are valid at that point are offered: a new transaction first begins, and a transaction whose
  commit or abort is in the schedule gets no further operations. **remove last** takes back the last operation.
- **Mode:** how the store validates a transaction at its commit: **method 1 (re-run reads, checksums)** runs
  every read of the transaction again and compares the checksums of the results; **method 2 (write sets)** checks
  the reads against the objects written by transactions that committed meanwhile; or **no validation** at all
  (anomalies such as write skew can happen). Without validation, transactions read their snapshot, and a write
  fails only while another transaction holds an uncommitted version of the object. This is weaker than snapshot
  isolation, which also stops a write to an object that another transaction committed after the writer's begin:
  try **lost update**, which commits here but would abort under snapshot isolation. With method 1, the
  transactions table shows the checksum of each read when it ran and the checksum the same read gives now; a
  commit now fails if they differ (red), unless the transaction wrote nothing (the store does not validate it
  then).
- **Run:** click an operation in the schedule to run everything up to and including it, or use **back** and
  **forward**. Operations that ran are blue, planned ones faint and in italics, and the one that ran last is framed
  (and yellow if it ran normally and is not a commit). A commit that succeeded is green; red marks where the store
  aborted a transaction, which then skips its remaining operations (grey); orange marks an error that stops the run.
  Once every transaction has ended, violet marks the read that returns something else in the order of the
  serialization points. The column **why** next to the transactions says what that read would return there, and why
  the store aborted a transaction.
- Below the schedule: the versions in the store, each transaction's status, its snapshot plus its own writes, what
  it read and wrote, and the verdict: whether the result is serializable and, unless it is in the order of the
  serialization points, one line per serial order that was replayed.

In [1]:
import sys, pathlib

# Put the repo root (the directory containing `system/`) on sys.path so
# `from system...` works whether this notebook is opened from its own
# folder or from the repo root.
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "system").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

In [2]:
from system.playground.mvcc_widgets import MvccPlayground

MvccPlayground().widget

## Things to try

- **lost update**, **write skew**: run them in each mode. Which transaction aborts with validation, and
  why is the result not serializable without validation?
- **phantom insert**, **concurrent delete**: without validation the result is serializable, but in another order
  than the store's. Is the abort with validation necessary?